In [1]:
import os
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [2]:
CLEAN_PATH = "../data/Chest Cancer Clean"

classes = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

In [3]:
image_paths = []
image_labels = []

for cls in classes:
    class_path = os.path.join(CLEAN_PATH, cls)

    for file in os.listdir(class_path):
        image_paths.append(os.path.join(class_path, file))
        image_labels.append(cls)

print("Total images:", len(image_paths))

Total images: 725


In [4]:
from sklearn.model_selection import train_test_split

image_paths = []
image_labels = []

for cls in classes:

    class_path = os.path.join(CLEAN_PATH, cls)

    for file in os.listdir(class_path):

        image_paths.append(
            os.path.join(class_path, file)
        )

        image_labels.append(cls)


X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    image_labels,
    test_size=0.30,
    stratify=image_labels,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 507
Validation: 109
Testing: 109


In [5]:
print("Training distribution:")
print(pd.Series(y_train).value_counts())

print("\nValidation distribution:")
print(pd.Series(y_val).value_counts())

print("\nTesting distribution:")
print(pd.Series(y_test).value_counts())

Training distribution:
Adenocarcinoma          233
Normal                  150
Large Cell Carcinoma    124
Name: count, dtype: int64

Validation distribution:
Adenocarcinoma          50
Normal                  33
Large Cell Carcinoma    26
Name: count, dtype: int64

Testing distribution:
Adenocarcinoma          50
Normal                  32
Large Cell Carcinoma    27
Name: count, dtype: int64


### Image Preprocessing

Medical images have different dimensions and color modes. Before training the CNN, all images are resized to a fixed resolution and converted to RGB.

The pixel values are normalized to the range [0, 1].

In [6]:
IMG_SIZE = (224, 224)

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = image.resize(IMG_SIZE)

    image = np.array(image) / 255.0

    return image

In [7]:
sample_image = preprocess_image(X_train[0])

print("Shape:", sample_image.shape)
print("Minimum:", sample_image.min())
print("Maximum:", sample_image.max())

Shape: (224, 224, 3)
Minimum: 0.0
Maximum: 1.0


### Data Augmentation

In [8]:
import tensorflow as tf

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1)
])

In [9]:
import tensorflow as tf
import sys

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)

Python: c:\Program Files\Python314\python.exe
TensorFlow: 2.22.0-rc0


In [11]:
sample = tf.convert_to_tensor(
    preprocess_image(X_train[0]),
    dtype=tf.float32
)

augmented = data_augmentation(
    tf.expand_dims(sample, axis=0),
    training=True
)

print("Original shape:", sample.shape)
print("Augmented shape:", augmented.shape)

Original shape: (224, 224, 3)
Augmented shape: (1, 224, 224, 3)


### A function to load images

In [12]:
def load_image(image_path, label):
    image = preprocess_image(image_path)
    
    return image, label

In [25]:
label_map = {
    "Adenocarcinoma": 0,
    "Large Cell Carcinoma": 1,
    "Normal": 2
}

y_train_encoded = [label_map[label] for label in y_train]
y_val_encoded = [label_map[label] for label in y_val]
y_test_encoded = [label_map[label] for label in y_test]

### Create TensorFlow datasets

In [26]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train_encoded)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val_encoded)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test_encoded)
)

In [27]:
def load_image(image_path, label):
    image = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32)

    return image, label

In [28]:
BATCH_SIZE = 16

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train_encoded)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val_encoded)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test_encoded)
)

train_dataset = train_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [29]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train_encoded)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val_encoded)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test_encoded)
)

In [30]:
train_dataset = train_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [31]:
def augment_image(image, label):
    image = data_augmentation(
        tf.expand_dims(image, axis=0),
        training=True
    )

    image = tf.squeeze(image, axis=0)

    return image, label

In [32]:
# Shuffle ONLY training data
train_dataset = train_dataset.shuffle(
    buffer_size=len(X_train),
    seed=42
)

# Augmentation ONLY for training
train_dataset = train_dataset.map(
    augment_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [33]:
# Batch
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(
    tf.data.AUTOTUNE
)

val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(
    tf.data.AUTOTUNE
)

test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(
    tf.data.AUTOTUNE
)

In [34]:
for images, labels in train_dataset.take(1):
    print("Images:", images.shape)
    print("Labels:", labels.shape)

Images: (16, 224, 224, 3)
Labels: (16,)


In [48]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)

class_weights = dict(
    zip(
        np.unique(y_train_encoded),
        class_weights
    )
)

class_weights

# Give more importance to Adenocarcinoma
class_weights[0] = class_weights[0] * 1.5

print(class_weights)

{np.int64(0): np.float64(1.0879828326180259), np.int64(1): np.float64(1.3629032258064515), np.int64(2): np.float64(1.1266666666666667)}


In [49]:
for class_name, class_id in label_map.items():
    print(class_name, ":", class_weights[class_id])

Adenocarcinoma : 1.0879828326180259
Large Cell Carcinoma : 1.3629032258064515
Normal : 1.1266666666666667


In [50]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

print("EfficientNetB0 imported successfully")

EfficientNetB0 imported successfully


In [51]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

### Build the EfficientNetB0 classifier

In [52]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax")
])

In [53]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [54]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

In [55]:
EPOCHS = 20

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr]
)

Epoch 1/20


C:\Users\shiva\AppData\Roaming\Python\Python314\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


32/32 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.4004 - loss: 1.3171 - val_accuracy: 0.6330 - val_loss: 0.8758 - learning_rate: 1.0000e-04
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 610ms/step - accuracy: 0.5720 - loss: 1.0558 - val_accuracy: 0.6972 - val_loss: 0.7209 - learning_rate: 1.0000e-04
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 618ms/step - accuracy: 0.6233 - loss: 0.9492 - val_accuracy: 0.6697 - val_loss: 0.6360 - learning_rate: 1.0000e-04
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 616ms/step - accuracy: 0.6864 - loss: 0.8170 - val_accuracy: 0.6972 - val_loss: 0.5916 - learning_rate: 1.0000e-04
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 613ms/step - accuracy: 0.7022 - loss: 0.8133 - val_accuracy: 0.7248 - val_loss: 0.5571 - learning_rate: 1.0000e-04
Epoch 6/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 655ms/step - accuracy: 0.7199 - loss: 0.7681 - val_accuracy: 0.7523 - val_loss: 0.5542 - learning_rate: 1.0000e-04
Epoch 7/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 610ms/step - accuracy: 0.7160 

In [56]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,542,638 (17.33 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 328,712 (1.25 MB)

In [57]:
import os

missing = [path for path in X_train + X_val + X_test if not os.path.exists(path)]

print("Missing images:", len(missing))

if missing:
    print(missing[:10])

Missing images: 0


In [58]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 500ms/step - accuracy: 0.9083 - loss: 0.3764
Test Loss: 0.3764135241508484
Test Accuracy: 0.9082568883895874


In [60]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

print(classification_report(
    y_true,
    y_pred,
    target_names=[
        "Adenocarcinoma",
        "Large Cell Carcinoma",
        "Normal"
    ]
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

                      precision    recall  f1-score   support

      Adenocarcinoma       0.92      0.88      0.90        50
Large Cell Carcinoma       0.85      0.85      0.85        27
              Normal       0.94      1.00      0.97        32

            accuracy                           0.91       109
           macro avg       0.90      0.91      0.91       109
        weighted avg       0.91      0.91      0.91       109

Confusion Matrix:
[[44  4  2]
 [ 4 23  0]
 [ 0  0 32]]


In [62]:
import os

os.makedirs("../models", exist_ok=True)

model.save("../models/chest_cancer_model.keras")

print("Chest model saved successfully.")

Chest model saved successfully.


In [63]:
model.save("../models/chest_cancer_model.keras")

In [69]:
import tensorflow as tf
import numpy as np

IMG_SIZE = (224, 224)

class_names = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

def predict_chest_image(image_path):

    image = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32)

    image = tf.expand_dims(image, axis=0)

    predictions = model.predict(image, verbose=0)[0]

    predicted_index = np.argmax(predictions)

    predicted_class = class_names[predicted_index]

    confidence = predictions[predicted_index]

    return predicted_class, confidence

In [70]:
test_image = "../data/Chest Cancer Clean/Adenocarcinoma/000000 (6).png"

predicted_class, confidence = predict_chest_image(test_image)

print("Prediction:", predicted_class)
print("Confidence:", round(float(confidence) * 100, 2), "%")

Prediction: Adenocarcinoma
Confidence: 61.23 %


In [73]:
import os

test_folders = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

for class_name in test_folders:

    folder_path = os.path.join(
        "../data/Chest Cancer Clean",
        class_name
    )

    test_file = os.listdir(folder_path)[0]
    test_image = os.path.join(folder_path, test_file)

    predicted_class, confidence = predict_chest_image(test_image)

    print("\nActual:", class_name)
    print("Image:", test_file)
    print("Prediction:", predicted_class)
    print("Confidence:", round(float(confidence) * 100, 2), "%")


Actual: Adenocarcinoma
Image: 000000 (6).png
Prediction: Adenocarcinoma
Confidence: 61.23 %

Actual: Large Cell Carcinoma
Image: 000002.png
Prediction: Adenocarcinoma
Confidence: 53.94 %

Actual: Normal
Image: 003828_02_01_174.png
Prediction: Normal
Confidence: 96.89 %


In [72]:
import os

base_path = "../data/Chest Cancer Clean"

for class_name in class_names:

    folder_path = os.path.join(base_path, class_name)

    files = os.listdir(folder_path)[:10]

    correct = 0

    print("\n==============================")
    print("Actual:", class_name)
    print("==============================")

    for file in files:

        image_path = os.path.join(folder_path, file)

        predicted_class, confidence = predict_chest_image(image_path)

        if predicted_class == class_name:
            correct += 1

        print(
            file,
            "→",
            predicted_class,
            round(float(confidence) * 100, 2), "%"
        )

    print(
        "Accuracy on these 10:",
        round(correct / len(files) * 100, 2), "%"
    )


Actual: Adenocarcinoma
000000 (6).png → Adenocarcinoma 61.23 %
000005 (3).png → Adenocarcinoma 55.52 %
000005 (9).png → Large Cell Carcinoma 51.45 %
000008 (10).png → Adenocarcinoma 59.05 %
000009 (3).png → Adenocarcinoma 70.64 %
000009 (7).png → Adenocarcinoma 66.84 %
000013 (4).png → Adenocarcinoma 89.65 %
000013 (8).png → Adenocarcinoma 83.29 %
000014 (7).png → Adenocarcinoma 51.54 %
000015 (10).png → Adenocarcinoma 50.66 %
Accuracy on these 10: 90.0 %

Actual: Large Cell Carcinoma
000002.png → Adenocarcinoma 53.94 %
000003 (3).png → Large Cell Carcinoma 77.92 %
000003 (4).png → Large Cell Carcinoma 71.84 %
000009 (3).png → Adenocarcinoma 56.38 %
000009 (4).png → Adenocarcinoma 51.84 %
000010.png → Large Cell Carcinoma 71.61 %
000015.png → Large Cell Carcinoma 58.44 %
000016 (3).png → Large Cell Carcinoma 49.89 %
000016 (4).png → Large Cell Carcinoma 52.69 %
000017.png → Large Cell Carcinoma 62.94 %
Accuracy on these 10: 70.0 %

Actual: Normal
003828_02_01_174.png → Normal 96.89 %
